In [7]:
from __future__ import annotations
from dataclasses import dataclass
from typing import Any, Callable, Dict, Mapping

import yaml

from quant_rotor.Hamiltonian_models.Dense.rotor_hamiltonian import hamiltonian_dense

# Build a wrapper.

In [ ]:
from __future__ import annotations

from typing import Any, Callable, Dict, Mapping, Union, Iterable, Tuple, List
import os, json, copy
import itertools

import numpy as np

try:
    import yaml
except ImportError:
    yaml = None

Hamiltonian = Any

# ---------------------------------------------------------------------
# Registries
# ---------------------------------------------------------------------
_BUILDERS: Dict[str, Callable[[Mapping[str, Any]], Hamiltonian]] = {}
_OPS: Dict[str, Callable[[Hamiltonian, Mapping[str, Any]], Hamiltonian]] = {}
_MEASURES: Dict[str, Callable[[Hamiltonian, Mapping[str, Any]], Any]] = {}


def register(model_type: str):
    def deco(fn):
        _BUILDERS[model_type] = fn
        return fn
    return deco


def register_op(op_type: str):
    def deco(fn):
        _OPS[op_type] = fn
        return fn
    return deco


def register_measure(measure_type: str):
    def deco(fn):
        _MEASURES[measure_type] = fn
        return fn
    return deco


# ---------------------------------------------------------------------
# Params loading: dict OR file path
# ---------------------------------------------------------------------
ParamsLike = Union[Mapping[str, Any], str, None]


def _load_mapping(path: str) -> Mapping[str, Any]:
    ext = os.path.splitext(path)[1].lower()
    with open(path, "r", encoding="utf-8") as f:
        if ext in (".yaml", ".yml"):
            if yaml is None:
                raise RuntimeError("pyyaml not installed but a .yaml/.yml params file was provided.")
            data = yaml.safe_load(f)
        elif ext == ".json":
            data = json.load(f)
        else:
            raise ValueError(f"Unsupported params file extension '{ext}'. Use .yaml/.yml or .json.")

    if not isinstance(data, dict):
        raise ValueError(f"Params file '{path}' must contain a mapping/dict at top level.")
    return data


def resolve_params(params: ParamsLike) -> Mapping[str, Any]:
    if params is None:
        return {}
    if isinstance(params, str):
        return _load_mapping(params)
    if isinstance(params, Mapping):
        return params
    raise TypeError(f"'params' must be a dict, a file path string, or null. Got: {type(params)}")


# ---------------------------------------------------------------------
# Core builder entry point
# ---------------------------------------------------------------------
def build_hamiltonian(cfg: Mapping[str, Any]) -> Hamiltonian:
    if "type" not in cfg:
        raise ValueError("Config must include key 'type'.")
    t = cfg["type"]
    if t not in _BUILDERS:
        raise ValueError(f"Unknown type '{t}'. Known types: {sorted(_BUILDERS)}")
    return _BUILDERS[t](cfg)


# ---------------------------------------------------------------------
# Operator pipeline
# ---------------------------------------------------------------------
def apply_ops(H: Hamiltonian, ops_cfg: List[Mapping[str, Any]]) -> Hamiltonian:
    for op_cfg in ops_cfg:
        op_type = op_cfg["type"]
        if op_type not in _OPS:
            raise ValueError(f"Unknown op '{op_type}'. Known ops: {sorted(_OPS)}")
        params = resolve_params(op_cfg.get("params"))
        H = _OPS[op_type](H, params)
    return H


@register("pipeline")
def _build_pipeline(cfg: Mapping[str, Any]) -> Hamiltonian:
    # cfg must have:
    #   hamiltonian: {type: ..., params: ...}
    #   ops: [{type: ..., params: ...}, ...]  (optional)
    if "hamiltonian" not in cfg:
        raise ValueError("pipeline config must include 'hamiltonian'.")

    ham_cfg = dict(cfg["hamiltonian"])
    ham_cfg["params"] = resolve_params(ham_cfg.get("params"))

    H = build_hamiltonian(ham_cfg)
    H = apply_ops(H, cfg.get("ops", []))
    return H


# ---------------------------------------------------------------------
# Measures: compute y from H
# ---------------------------------------------------------------------
def compute_measure(H: Hamiltonian, measure_cfg: Mapping[str, Any]) -> Any:
    mtype = measure_cfg["type"]
    if mtype not in _MEASURES:
        raise ValueError(f"Unknown measure '{mtype}'. Known measures: {sorted(_MEASURES)}")
    params = resolve_params(measure_cfg.get("params"))
    return _MEASURES[mtype](H, params)


# ---------------------------------------------------------------------
# Plot sweep helpers
# ---------------------------------------------------------------------
def set_by_path(d: dict, path: str, value: Any) -> None:
    keys = path.split(".")
    cur = d
    for k in keys[:-1]:
        cur = cur[k]
    cur[keys[-1]] = value


def values_from_axis(axis_cfg: dict) -> List[Any]:
    if "values" in axis_cfg:
        return list(axis_cfg["values"])
    if "linspace" in axis_cfg:
        ls = axis_cfg["linspace"]
        return np.linspace(ls["start"], ls["stop"], ls["num"]).tolist()
    if "arange" in axis_cfg:
        ar = axis_cfg["arange"]
        return np.arange(ar["start"], ar["stop"], ar["step"]).tolist()
    raise ValueError("Axis must have 'values', 'linspace', or 'arange'.")


def run_series_plot(cfg: dict) -> List[Tuple[Any, List[float], List[Any]]]:
    """
    General structure:
      - x = cfg["plot"]["x"] (sweep over x-axis parameter)
      - series = cfg["plot"]["series"] (each value -> separate curve)
      - build pipeline -> compute measure -> y

    Returns:
      curves = [(series_value, x_values, y_values), ...]
    """

    base = cfg["base"]  # contains at least "hamiltonian" + optionally "ops"
    x_path = cfg["plot"]["x"]["path"]
    x_vals = values_from_axis(cfg["plot"]["x"])

    s_path = cfg["plot"]["series"]["path"]
    s_vals = values_from_axis(cfg["plot"]["series"])

    ops_cfg = cfg.get("pipeline", {}).get("ops", [])  # optional shared ops
    measure_cfg = cfg["measure"]

    curves = []

    for s in s_vals:
        y_vals = []
        for x in x_vals:
            run_cfg = copy.deepcopy(base)

            # patch series and x parameter
            set_by_path(run_cfg, s_path, s)
            set_by_path(run_cfg, x_path, x)

            # Build H via pipeline (recommended) OR direct
            # Here we build a pipeline config on the fly:
            pipe_cfg = {
                "type": "pipeline",
                "hamiltonian": run_cfg["hamiltonian"],
                "ops": ops_cfg,
            }
            H = build_hamiltonian(pipe_cfg)

            y = compute_measure(H, measure_cfg)
            y_vals.append(y)

        curves.append((s, x_vals, y_vals))

    return curves


# ---------------------------------------------------------------------
# EXAMPLE OPS (edit to match your backends)
# ---------------------------------------------------------------------
@register_op("symmetrize")
def op_symmetrize(H, p):
    # works for numpy arrays; adjust if sparse
    return 0.5 * (H + H.conj().T)


@register_op("shift")
def op_shift(H, p):
    mu = float(p.get("mu", 0.0))
    # dense version:
    return H + mu * np.eye(H.shape[0], dtype=H.dtype)


# ---------------------------------------------------------------------
# EXAMPLE MEASURES
# ---------------------------------------------------------------------
@register_measure("ground_energy")
def measure_ground_energy(H, p):
    # Dense only. If H is sparse, use scipy.sparse.linalg.eigsh
    evals = np.linalg.eigvalsh(H)
    return float(evals[0])


@register_measure("gap")
def measure_gap(H, p):
    evals = np.linalg.eigvalsh(H)
    return float(evals[1] - evals[0])


# ---------------------------------------------------------------------
# YOUR BUILDERS (plug in your real constructors here)
# ---------------------------------------------------------------------
def hamiltonian_dense(state, site, g_val, psi_twist, lambda_val, D, Double, periodic, field):
    # Placeholder: replace with your real function.
    # Must return a square matrix (numpy ndarray) for the example ops/measures.
    dim = int(state)
    rng = np.random.default_rng(0)
    A = rng.normal(size=(dim, dim))
    H = 0.5 * (A + A.T)
    return H


@register("rotor")
def _build_rotor_1d(cfg: Mapping[str, Any]) -> Hamiltonian:
    p = resolve_params(cfg.get("params"))

    state = int(p["state"])
    site = int(p["site"])
    g_val = float(p["g_val"])
    psi_twist = float(p["psi_twist"])
    lambda_val = float(p["lambda_val"])
    D = float(p["D"])
    Double = bool(p["Double"])
    periodic = bool(p["periodic"])
    field = bool(p["field"])

    return hamiltonian_dense(state, site, g_val, psi_twist, lambda_val, D, Double, periodic, field)


@register("heisenberg_1d")
def _build_heisenberg_1d(cfg: Mapping[str, Any]) -> Hamiltonian:
    p = resolve_params(cfg.get("params"))

    state = int(p["state"])
    site = int(p["site"])
    g_val = float(p["g_val"])
    psi_twist = float(p["psi_twist"])
    lambda_val = float(p["lambda_val"])
    D = float(p["D"])
    Double = bool(p["Double"])
    periodic = bool(p["periodic"])
    field = bool(p["field"])

    return hamiltonian_dense(state, site, g_val, psi_twist, lambda_val, D, Double, periodic, field)

**Objects**

In [ ]:
@register("rotor")
def _build_rotor_1d(cfg: Mapping[str, Any]) -> Hamiltonian:

    p = cfg["params"]
    state, site = int(p['state']), int(p['site'])
    g_val, psi_twist, lambda_val, D = float(p['g_val']), float(p['psi_twist']), float(p['lambda_val']), float(p['D'])
    Double, periodic, field = bool(p['Double']), bool(p['periodic']), bool(p['field'])

    return hamiltonian_dense(state, site, g_val, psi_twist, lambda_val, D, Double, periodic, field)

@register("heisenberg_1d")
def _build_heisenberg_1d(cfg: Mapping[str, Any]) -> Hamiltonian:

    p = cfg["params"]
    state, site = int(p['state']), int(p['site'])
    g_val, psi_twist, lambda_val, D = float(p['g_val']), float(p['psi_twist']), float(p['lambda_val']), float(p['D'])
    Double, periodic, field = bool(p['Double']), bool(p['periodic']), bool(p['field'])

    return hamiltonian_dense(state, site, g_val, psi_twist, lambda_val, D, Double, periodic, field)

**Operators**

In [9]:
cfg = yaml.safe_load(open("Config/rotor_ham_config.yaml"))
H = build_hamiltonian(cfg)